# BATADAL RAG explanation layer, three detectors

Reuses the data loading, episode split, Random Forest feature pipeline, sensor graphs,
forecaster windowing, GDN/TGCN model classes, and training/evaluation utilities from
batadal_final.ipynb. Two scope reductions from that notebook: Random Forest Protocol A and
the Isolation Forest baseline are omitted, and GDN/TGCN are trained on the Granger graph only,
since the explanation layer only needs one detection pipeline per model to build on.

ATT_FLAG is used as it appears in dataset04. ATTACK_ID, built from the documented attack
duration table, is used only for splitting and for the knowledge graph's attack descriptions,
and never to redefine the label. Attacks 2 and 7 have zero officially labeled hours despite
spanning 24 and 110 window hours, since BATADAL's concealment periods are unlabeled by design.

Sections 9 to 13 (attack pattern library, Neo4j knowledge graph, retrieval, Groq explanation
generation, quality check) are new work on top of the detection pipeline.

## 0. Setup

Groq and Neo4j credentials are hardcoded here rather than pulled from Colab Secrets so the
notebook runs standalone without needing access to a specific account's secret store.

In [1]:
!pip install -q neo4j groq

In [2]:
import itertools
import warnings
warnings.filterwarnings("ignore")

import logging
logging.getLogger("neo4j").setLevel(logging.ERROR)
logging.getLogger("neo4j.notifications").setLevel(logging.ERROR)

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (f1_score, precision_score, recall_score, roc_auc_score,
                              average_precision_score, precision_recall_curve,
                              classification_report)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from statsmodels.tsa.stattools import grangercausalitytests

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

WINDOW_SIZE = 24
EPOCHS_DEFAULT = 40
PATIENCE_DEFAULT = 6

GROQ_API_KEY = "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"
NEO4J_URI = "neo4j+s://156b70ec.databases.neo4j.io"
NEO4J_USERNAME = "156b70ec"
NEO4J_PASSWORD = "o4l3vjwcVMN5EfbpnJa4_VI218ZOE22iddOvQVrv4sE"
NEO4J_DATABASE = "156b70ec"

DATASET_NAME = "BATADAL"
RESET_DATASET = True

print(f"device: {device}")

device: cpu


## 1. Data and labels

`ATT_FLAG` is the file's own column, `ATTACK_ID` is
derived from the documented duration table purely for splitting and reporting.

In [3]:
DATA_DIR = "."

df03 = pd.read_csv(f"{DATA_DIR}/BATADAL_dataset03.csv")
df03.columns = df03.columns.str.strip()
df03["DATETIME"] = pd.to_datetime(df03["DATETIME"].str.strip(), format="%d/%m/%y %H")
df03["ATT_FLAG"] = np.where(df03["ATT_FLAG"] == -999, 0, df03["ATT_FLAG"])
df03["ATTACK_ID"] = 0
assert df03["ATT_FLAG"].sum() == 0

df04 = pd.read_csv(f"{DATA_DIR}/BATADAL_dataset04.csv")
df04.columns = df04.columns.str.strip()
df04["DATETIME"] = pd.to_datetime(df04["DATETIME"].str.strip(), format="%d/%m/%y %H")
df04["ATT_FLAG"] = np.where(df04["ATT_FLAG"] == 1, 1, 0)

ATTACK_DURATION_WINDOWS = {
    1: ("2016-09-13 23:00", "2016-09-16 00:00"),
    2: ("2016-09-26 11:00", "2016-09-27 10:00"),
    3: ("2016-10-09 09:00", "2016-10-11 20:00"),
    4: ("2016-10-29 19:00", "2016-11-02 16:00"),
    5: ("2016-11-26 17:00", "2016-11-29 04:00"),
    6: ("2016-12-06 07:00", "2016-12-10 04:00"),
    7: ("2016-12-14 15:00", "2016-12-19 04:00"),
}
df04["ATTACK_ID"] = 0
for attack_id, (start, end) in ATTACK_DURATION_WINDOWS.items():
    df04.loc[(df04["DATETIME"] >= start) & (df04["DATETIME"] <= end), "ATTACK_ID"] = attack_id

status_cols = [c for c in df03.columns if c.startswith("S_")]
df03 = df03.drop(columns=status_cols)
df04 = df04.drop(columns=status_cols)

FEATURE_COLS = [c for c in df03.columns if c not in ("DATETIME", "ATT_FLAG", "ATTACK_ID")]
combined = pd.concat([df03, df04], ignore_index=True).sort_values("DATETIME").reset_index(drop=True)
y_all = combined["ATT_FLAG"].values
attack_id_all = combined["ATTACK_ID"].values
row_idx = np.arange(len(combined))

print(f"Combined dataset: {combined.shape[0]} rows, {len(FEATURE_COLS)} sensor features")
print(f"Positively labeled attack rows: {y_all.sum()} ({y_all.mean() * 100:.2f}%)")
for attack_id in range(1, 8):
    mask = attack_id_all == attack_id
    print(f"  Attack {attack_id}: {y_all[mask].sum()} labeled hours out of {mask.sum()} window hours")

Combined dataset: 12938 rows, 31 sensor features
Positively labeled attack rows: 219 (1.69%)
  Attack 1: 42 labeled hours out of 50 window hours
  Attack 2: 0 labeled hours out of 24 window hours
  Attack 3: 60 labeled hours out of 60 window hours
  Attack 4: 37 labeled hours out of 94 window hours
  Attack 5: 7 labeled hours out of 60 window hours
  Attack 6: 73 labeled hours out of 94 window hours
  Attack 7: 0 labeled hours out of 110 window hours


### Official attack descriptions and PLC relationships

`plc_edges` records the PLC-level relationships stated directly in the attack descriptions, used later
to add genuine control-logic structure to the graph rather than relying on correlation or causal
inference alone.

In [4]:
ATTACK_DESCRIPTIONS = {
    1: "Attacker changes L_T7 thresholds (which control PU10/PU11) by altering the SCADA transmission to "
       "PLC9, causing low levels in T7. Concealed via a replay attack on L_T7.",
    2: "Same mechanism as Attack #1 (L_T7 threshold manipulation via PLC9). Concealment extended: replay "
       "attack also covers PU10/PU11 flow and status.",
    3: "Attack alters L_T1 readings sent by PLC2 to PLC1, which then reads a constant low level and keeps "
       "pumps PU1/PU2 ON, causing an overflow in T1. Concealed with a polyline offset applied to the L_T1 "
       "increase.",
    4: "Same mechanism as Attack #3 (L_T1 manipulation via PLC2 to PLC1). Concealed via a replay attack "
       "covering L_T1, PU1/PU2 flow and status, and pressure at the pumps' outlet.",
    5: "Working speed of PU7 reduced to 0.9 of nominal, causing lower water levels in T4. No SCADA "
       "concealment recorded.",
    6: "Same mechanism as Attack #5 but PU7 speed reduced further, to 0.7 of nominal. The resulting L_T4 "
       "drop is concealed with a replay attack.",
    7: "Same mechanism as Attack #6 (PU7 speed reduction). Concealed via a replay attack on L_T1, as well "
       "as PU1/PU2 flow and status.",
}

SIBLING_ATTACKS = {2: 1, 4: 3, 6: 5, 7: 6}

plc_edges = [
    ("PLC9", "RECEIVES_READING", "L_T7"),
    ("L_T7", "THRESHOLD_CONTROLS", "F_PU10"),
    ("L_T7", "THRESHOLD_CONTROLS", "F_PU11"),
    ("PLC2", "READS", "L_T1"),
    ("PLC2", "SENDS_READING_TO", "PLC1"),
    ("PLC1", "CONTROLS", "F_PU1"),
    ("PLC1", "CONTROLS", "F_PU2"),
]

print(f"{len(ATTACK_DESCRIPTIONS)} attack descriptions, {len(plc_edges)} PLC-level edges")

7 attack descriptions, 7 PLC-level edges


In [5]:
def node_type_for_tag(tag):
    if tag.startswith("L_"):
        return "tank"
    if "PU" in tag:
        return "pump"
    if tag.startswith("F_V"):
        return "valve"
    if tag.startswith("P_"):
        return "junction"
    return "sensor"

def asset_id_for_tag(tag):
    return tag.replace("F_", "").replace("L_", "").replace("P_", "")

ASSET_TYPES = {tag: node_type_for_tag(tag) for tag in FEATURE_COLS}
ASSET_IDS = {tag: asset_id_for_tag(tag) for tag in FEATURE_COLS}
print(pd.Series(ASSET_TYPES).value_counts())

junction    12
pump        11
tank         7
valve        1
Name: count, dtype: int64


## 2. Episode split

Train on attacks 1 to 3 plus all normal
data, tune the Random Forest threshold on attack 4, evaluate once on attacks 5 to 7. No attack ever
appears on both sides.

In [6]:
TRAIN_ATTACKS = {1, 2, 3}
VAL_ATTACK = {4}
TEST_ATTACKS = {5, 6, 7}

episode_train_idx = row_idx[np.isin(attack_id_all, list(TRAIN_ATTACKS) + [0])]
episode_val_idx = row_idx[np.isin(attack_id_all, list(VAL_ATTACK))]
episode_test_idx = row_idx[np.isin(attack_id_all, list(TEST_ATTACKS))]

print(f"episode_split train: {len(episode_train_idx)} rows ({y_all[episode_train_idx].sum()} labeled attack hours)")
print(f"episode_split val (attack 4): {len(episode_val_idx)} rows ({y_all[episode_val_idx].sum()} labeled attack hours)")
print(f"episode_split test (attacks 5-7): {len(episode_test_idx)} rows ({y_all[episode_test_idx].sum()} labeled attack hours)")

episode_split train: 12580 rows (102 labeled attack hours)
episode_split val (attack 4): 94 rows (37 labeled attack hours)
episode_split test (attacks 5-7): 264 rows (80 labeled attack hours)


## 3. Random Forest feature engineering

Cross-sensor residuals are fit on `dataset03` only, so
they carry no label information. Feature selection is refit on the training partition only.

In [7]:
X03 = df03[FEATURE_COLS].values
residual_models = {}
for i, col in enumerate(FEATURE_COLS):
    other_idx = [j for j in range(len(FEATURE_COLS)) if j != i]
    reg = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=RANDOM_STATE, n_jobs=2)
    reg.fit(X03[:, other_idx], X03[:, i])
    residual_models[col] = (reg, other_idx)

def add_residual_features(df, feature_cols, models):
    X = df[feature_cols].values
    resid = np.zeros_like(X, dtype=np.float32)
    for i, col in enumerate(feature_cols):
        model, other_idx = models[col]
        resid[:, i] = X[:, i] - model.predict(X[:, other_idx])
    resid_df = pd.DataFrame(resid, columns=[f"{c}_resid" for c in feature_cols], index=df.index)
    return pd.concat([df, resid_df], axis=1)

combined = add_residual_features(combined, FEATURE_COLS, residual_models)
RESID_COLS = [f"{c}_resid" for c in FEATURE_COLS]
print(f"Trained {len(residual_models)} cross-sensor residual regressors on normal-operation data")

Trained 31 cross-sensor residual regressors on normal-operation data


In [8]:
def add_temporal_features(df, cols, lags=(1, 3), roll=5):
    out = df.copy()
    new_cols = {}
    for col in cols:
        for lag in lags:
            new_cols[f"{col}_lag{lag}"] = out[col].shift(lag)
        new_cols[f"{col}_d1"] = out[col].diff(1)
        new_cols[f"{col}_mean{roll}"] = out[col].rolling(roll).mean()
        new_cols[f"{col}_std{roll}"] = out[col].rolling(roll).std()
    out = pd.concat([out, pd.DataFrame(new_cols, index=out.index)], axis=1)
    return out.ffill().bfill()

def select_features_and_engineer(train_idx_for_selection):
    X_sel = combined.loc[train_idx_for_selection, FEATURE_COLS].values
    y_sel = combined.loc[train_idx_for_selection, "ATT_FLAG"].values
    selector = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=2)
    selector.fit(X_sel, y_sel)
    importances = pd.Series(selector.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)
    top_features = importances.head(10).index.tolist()

    eng = add_temporal_features(combined[["DATETIME", "ATT_FLAG"] + FEATURE_COLS], top_features)
    eng = pd.concat([eng, combined[RESID_COLS]], axis=1)
    eng_cols = [c for c in eng.columns if c not in ("DATETIME", "ATT_FLAG")]
    return eng, eng_cols, top_features

## 4. Random Forest, Protocol B (attack-disjoint, no leakage)

Class-weighted rather than SMOTE-balanced, hyperparameters
selected via three-fold cross-validation on the training partition only, threshold chosen on attack 4
by maximising F1 over the precision-recall curve, applied once to attacks 5 to 7.

In [9]:
eng_b, eng_cols_b, top_features_b = select_features_and_engineer(episode_train_idx)

X_b = eng_b[eng_cols_b].values
X_train_raw = X_b[episode_train_idx]
y_train = y_all[episode_train_idx]
X_val_raw = X_b[episode_val_idx]
y_val = y_all[episode_val_idx]
X_test_raw = X_b[episode_test_idx]
y_test = y_all[episode_test_idx]

scaler_b = StandardScaler()
X_train = scaler_b.fit_transform(X_train_raw)
X_val = scaler_b.transform(X_val_raw)
X_test = scaler_b.transform(X_test_raw)

param_grid = {
    "n_estimators": [200, 400],
    "max_depth": [8, 15, None],
    "min_samples_leaf": [1, 3, 5],
}
rf_base = RandomForestClassifier(class_weight="balanced", random_state=RANDOM_STATE, n_jobs=2)
grid = GridSearchCV(rf_base, param_grid, scoring="f1", cv=3, n_jobs=2)
grid.fit(X_train, y_train)
rf_model = grid.best_estimator_
print("Best params (3-fold CV on training partition only):", grid.best_params_)

val_probs = rf_model.predict_proba(X_val)[:, 1]
precisions, recalls, thresholds = precision_recall_curve(y_val, val_probs)
f1_curve = np.where((precisions + recalls) > 0, 2 * precisions * recalls / (precisions + recalls + 1e-10), 0)
rf_threshold = thresholds[np.argmax(f1_curve[:-1])] if len(thresholds) > 0 else 0.5

test_probs = rf_model.predict_proba(X_test)[:, 1]
test_preds = (test_probs >= rf_threshold).astype(int)

rf_results = {
    "precision": precision_score(y_test, test_preds, zero_division=0),
    "recall": recall_score(y_test, test_preds, zero_division=0),
    "f1": f1_score(y_test, test_preds, zero_division=0),
    "roc_auc": roc_auc_score(y_test, test_probs) if y_test.sum() > 0 else float("nan"),
    "pr_auc": average_precision_score(y_test, test_probs) if y_test.sum() > 0 else float("nan"),
}
print("\nRF Protocol B (attack-disjoint, primary):")
for k, v in rf_results.items():
    print(f"  {k:10s}: {v:.4f}")
print()
print(classification_report(y_test, test_preds, target_names=["Normal", "Attack"]))

rf_train_median, rf_train_mad = None, None
_med = np.median(X03, axis=0)
_mad = np.median(np.abs(X03 - _med), axis=0) + 1e-6
rf_train_median, rf_train_mad = _med, _mad

Best params (3-fold CV on training partition only): {'max_depth': 8, 'min_samples_leaf': 5, 'n_estimators': 200}

RF Protocol B (attack-disjoint, primary):
  precision : 0.2097
  recall    : 0.1625
  f1        : 0.1831
  roc_auc   : 0.4691
  pr_auc    : 0.2704

              precision    recall  f1-score   support

      Normal       0.67      0.73      0.70       184
      Attack       0.21      0.16      0.18        80

    accuracy                           0.56       264
   macro avg       0.44      0.45      0.44       264
weighted avg       0.53      0.56      0.54       264



## 5. Sensor graphs

Both graphs are fit on `dataset03` (normal only, no label
leakage). Correlation keeps the top ten strongest edges per node, Granger keeps the top ten edges per
node by causal strength at lag 3, falling back to correlation for any node with no statistically
significant causal parent. Only the Granger graph is used to train GDN and TGCN below.

In [10]:
def build_correlation_edge_index(X, topk=10):
    n = X.shape[1]
    corr = np.nan_to_num(np.corrcoef(X, rowvar=False), nan=0.0)
    np.fill_diagonal(corr, 0.0)
    abs_corr = np.abs(corr)
    src, dst = [], []
    for i in range(n):
        for j in np.argsort(-abs_corr[i])[:topk]:
            src.append(j)
            dst.append(i)
    return np.array([src, dst], dtype=np.int64)

corr_edge_index_np = build_correlation_edge_index(X03, topk=10)
print(f"Correlation graph: {len(FEATURE_COLS)} nodes, {corr_edge_index_np.shape[1]} directed edges")

Correlation graph: 31 nodes, 310 directed edges


In [11]:
GRANGER_MAXLAG = 3
GRANGER_ALPHA = 0.05
GRANGER_TOPK = 10

def build_granger_edge_index(X_normal, maxlag=GRANGER_MAXLAG, alpha=GRANGER_ALPHA, topk=GRANGER_TOPK):
    n = X_normal.shape[1]
    pvals = np.full((n, n), np.nan)
    n_failed = 0
    for i, j in itertools.permutations(range(n), 2):
        try:
            res = grangercausalitytests(X_normal[:, [i, j]], maxlag=maxlag)
            pvals[i, j] = min(res[k][0]["ssr_ftest"][1] for k in range(1, maxlag + 1))
        except Exception:
            n_failed += 1

    corr = np.nan_to_num(np.corrcoef(X_normal, rowvar=False), nan=0.0)
    np.fill_diagonal(corr, 0.0)
    abs_corr = np.abs(corr)

    src, dst, n_fallback = [], [], 0
    for i in range(n):
        row = pvals[:, i].copy()
        row[i] = np.nan
        significant = np.where(row < alpha)[0]
        if len(significant) == 0:
            n_fallback += 1
            chosen = np.argsort(-abs_corr[i])[:topk]
        else:
            chosen = significant[np.argsort(row[significant])][:topk]
        for j in chosen:
            src.append(j)
            dst.append(i)
    print(f"Granger tests: {n * (n - 1)} pairs, {n_failed} undefined (constant-value sources)")
    print(f"Nodes with no significant causal parent (correlation fallback): {n_fallback}/{n}")
    return np.array([src, dst], dtype=np.int64), pvals

granger_edge_index_np, granger_pvals = build_granger_edge_index(X03)
granger_edge_index = torch.from_numpy(granger_edge_index_np).to(device)
print(f"Granger-causal graph: {len(FEATURE_COLS)} nodes, {granger_edge_index_np.shape[1]} directed edges")

Granger tests: 930 pairs, 174 undefined (constant-value sources)
Nodes with no significant causal parent (correlation fallback): 3/31
Granger-causal graph: 31 nodes, 303 directed edges


## 6. Forecaster windowing

Windows use `WINDOW_SIZE` hours of history to forecast the
next hour. Training windows are normal rows only, excluding anything within 48 hours of the validation
or test attack episodes. Validation windows surround attack 4, test windows surround attacks 5 to 7.

In [12]:
X_raw_all = combined[FEATURE_COLS].values
n_nodes = X_raw_all.shape[1]

def get_window_ends_for_ids(attack_id_arr, target_ids, window_size, n_rows, surrounding_hours=48):
    valid_ends = np.arange(window_size - 1, n_rows)
    is_target_attack = np.isin(attack_id_arr[valid_ends], list(target_ids))

    target_positions = np.where(np.isin(attack_id_arr, list(target_ids)))[0]
    near_mask = np.zeros(len(valid_ends), dtype=bool)
    if len(target_positions) > 0:
        for pos in target_positions:
            lo, hi = max(0, pos - surrounding_hours), min(n_rows - 1, pos + surrounding_hours)
            near_mask |= (valid_ends >= lo) & (valid_ends <= hi)

    return valid_ends[is_target_attack | (near_mask & (attack_id_arr[valid_ends] == 0))]

def get_normal_training_ends(attack_id_arr, exclude_ids, window_size, n_rows):
    valid_ends = np.arange(window_size - 1, n_rows)
    normal_ends = valid_ends[attack_id_arr[valid_ends] == 0]
    excluded_ends = set(get_window_ends_for_ids(attack_id_arr, exclude_ids, window_size, n_rows).tolist())
    return np.array([e for e in normal_ends if e not in excluded_ends])

gdn_train_ends = get_normal_training_ends(attack_id_all, VAL_ATTACK | TEST_ATTACKS, WINDOW_SIZE, len(X_raw_all))
gdn_val_ends = get_window_ends_for_ids(attack_id_all, VAL_ATTACK, WINDOW_SIZE, len(X_raw_all))
gdn_test_ends = get_window_ends_for_ids(attack_id_all, TEST_ATTACKS, WINDOW_SIZE, len(X_raw_all))

print(f"Training windows (normal only): {len(gdn_train_ends)}")
print(f"Validation windows (attack 4 + surrounding normal): {len(gdn_val_ends)} ({y_all[gdn_val_ends].sum()} labeled attack hours)")
print(f"Test windows (attacks 5-7 + surrounding normal): {len(gdn_test_ends)} ({y_all[gdn_test_ends].sum()} labeled attack hours)")

Training windows (normal only): 12039
Validation windows (attack 4 + surrounding normal): 190 (37 labeled attack hours)
Test windows (attacks 5-7 + surrounding normal): 552 (80 labeled attack hours)


In [13]:
scaler_gdn = StandardScaler()
scaler_gdn.fit(X_raw_all[np.array([e for e in gdn_train_ends])])
X_scaled_gdn = scaler_gdn.transform(X_raw_all).astype(np.float32)

## 7. Models

GDN uses graph attention restricted to the Granger graph's
edges, followed by a GRU over the window and a linear forecast head, reading the GRU's final hidden
state. TGCN uses fixed symmetrically-normalised graph convolution per timestep on the same graph,
followed by the same GRU and head. Both are trained with MSE on normal windows only, anomaly score is
forecast error.

In [14]:
class ForecastWindowDataset(Dataset):
    def __init__(self, X, end_indices, window_size):
        self.X, self.ends, self.w = X, end_indices, window_size

    def __len__(self):
        return len(self.ends)

    def __getitem__(self, i):
        e = self.ends[i]
        window = self.X[e - self.w + 1: e]
        target = self.X[e]
        return torch.from_numpy(window), torch.from_numpy(target)

class GraphAttentionLayer(nn.Module):
    def __init__(self, in_dim, out_dim, edge_index, n_nodes):
        super().__init__()
        self.edge_index, self.n_nodes = edge_index, n_nodes
        self.lin = nn.Linear(in_dim, out_dim)
        self.attn_src = nn.Linear(out_dim, 1, bias=False)
        self.attn_dst = nn.Linear(out_dim, 1, bias=False)

    def forward(self, x):
        h = self.lin(x)
        src, dst = self.edge_index[0], self.edge_index[1]
        e = F.leaky_relu(self.attn_src(h).squeeze(-1)[:, src] + self.attn_dst(h).squeeze(-1)[:, dst])
        e_max = e.max(dim=1, keepdim=True)[0].detach()
        e_exp = torch.exp(e - e_max)
        denom = torch.zeros(x.size(0), self.n_nodes, device=x.device)
        denom.scatter_add_(1, dst.unsqueeze(0).expand(x.size(0), -1), e_exp)
        alpha = e_exp / denom.clamp_min(1e-8)[:, dst]
        out = torch.zeros(x.size(0), self.n_nodes, h.size(-1), device=x.device)
        weighted = h[:, src] * alpha.unsqueeze(-1)
        out.scatter_add_(1, dst.unsqueeze(0).unsqueeze(-1).expand(x.size(0), -1, h.size(-1)), weighted)
        return F.relu(out)

class GDNForecaster(nn.Module):
    def __init__(self, n_nodes, edge_index, hidden_dim=16, gru_hidden=64):
        super().__init__()
        self.n_nodes = n_nodes
        self.gat = GraphAttentionLayer(1, hidden_dim, edge_index, n_nodes)
        self.gru = nn.GRU(input_size=n_nodes * hidden_dim, hidden_size=gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, n_nodes)

    def forward(self, window):
        B, T, N = window.shape
        x = window.reshape(B * T, N, 1)
        h = self.gat(x).reshape(B, T, -1)
        gru_out, _ = self.gru(h)
        return self.head(gru_out[:, -1, :])

def build_normalized_adjacency(edge_index_np, n_nodes, add_self_loops=True):
    A = np.zeros((n_nodes, n_nodes), dtype=np.float32)
    A[edge_index_np[1], edge_index_np[0]] = 1.0
    if add_self_loops:
        A += np.eye(n_nodes, dtype=np.float32)
    deg = A.sum(axis=1)
    deg_inv_sqrt = np.power(deg, -0.5, where=deg > 0)
    deg_inv_sqrt[deg == 0] = 0.0
    D_inv_sqrt = np.diag(deg_inv_sqrt)
    return torch.from_numpy(D_inv_sqrt @ A @ D_inv_sqrt).to(device)

adj_norm_granger = build_normalized_adjacency(granger_edge_index_np, n_nodes)

class TGCNForecaster(nn.Module):
    def __init__(self, n_nodes, adj_norm, hidden_dim=16, gru_hidden=64):
        super().__init__()
        self.n_nodes = n_nodes
        self.adj_norm = adj_norm
        self.gcn_lin = nn.Linear(1, hidden_dim)
        self.gru = nn.GRU(input_size=n_nodes * hidden_dim, hidden_size=gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, n_nodes)

    def forward(self, window):
        B, T, N = window.shape
        x = window.reshape(B * T, N, 1)
        h = self.gcn_lin(x)
        h = torch.einsum("ij,bjd->bid", self.adj_norm, h)
        h = F.relu(h).reshape(B, T, -1)
        gru_out, _ = self.gru(h)
        return self.head(gru_out[:, -1, :])

## 8. Training and evaluation utilities

`evaluate_forecaster` thresholds at the maximum forecast
error seen on the validation windows (attack 4 plus its surrounding normal buffer), a one-sided rule
that needs no separate precision-recall sweep since the validation windows already contain a realistic
mix of normal and attack behaviour. Point-adjusted metrics follow the usual time-series anomaly
detection convention: a whole attack episode counts as detected if any of its labeled hours were
flagged.

In [15]:
def train_forecaster(model_cls, model_kwargs, train_ends, val_ends, X_scaled, window_size,
                      epochs=EPOCHS_DEFAULT, patience=PATIENCE_DEFAULT, lr=1e-3, batch_size=64):
    train_ds = ForecastWindowDataset(X_scaled, train_ends, window_size)
    val_ds = ForecastWindowDataset(X_scaled, val_ends, window_size)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

    model = model_cls(**model_kwargs).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    criterion = nn.MSELoss()

    best_val_loss, best_state, no_improve = float("inf"), None, 0
    for epoch in range(epochs):
        model.train()
        for window, target in train_loader:
            window, target = window.to(device), target.to(device)
            optimizer.zero_grad()
            pred = model(window)
            loss = criterion(pred, target)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        model.eval()
        val_losses = []
        with torch.no_grad():
            for window, target in val_loader:
                pred = model(window.to(device))
                val_losses.append(F.mse_loss(pred, target.to(device), reduction="none").mean(dim=1).cpu().numpy())
        val_loss = np.concatenate(val_losses).mean() if val_losses else float("inf")

        if epoch % 5 == 0 or epoch == epochs - 1:
            print(f"  epoch {epoch + 1}/{epochs} | val forecast MSE {val_loss:.5f}")

        if val_loss < best_val_loss:
            best_val_loss, best_state, no_improve = val_loss, {k: v.clone() for k, v in model.state_dict().items()}, 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f"  early stopping at epoch {epoch + 1}")
                break

    model.load_state_dict(best_state)
    return model

def compute_window_scores(model, end_indices, X_scaled, window_size, batch_size=256):
    ds = ForecastWindowDataset(X_scaled, end_indices, window_size)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False)
    model.eval()
    scores = []
    with torch.no_grad():
        for window, target in loader:
            pred = model(window.to(device))
            mse_per_row = F.mse_loss(pred, target.to(device), reduction="none").mean(dim=1)
            scores.append(mse_per_row.cpu().numpy())
    return np.concatenate(scores)

def point_adjusted_labels_and_preds(y_true, preds, attack_id_arr_subset):
    adjusted_preds = preds.copy()
    for attack_id in np.unique(attack_id_arr_subset):
        if attack_id == 0:
            continue
        mask = attack_id_arr_subset == attack_id
        if y_true[mask].sum() > 0 and preds[mask].sum() > 0:
            adjusted_preds[mask] = y_true[mask]
    return y_true, adjusted_preds

def evaluate_forecaster(model, val_ends, test_ends, X_scaled, y_all, attack_id_all, window_size, label=""):
    val_scores = compute_window_scores(model, val_ends, X_scaled, window_size)
    threshold_max_val = val_scores.max()

    test_scores = compute_window_scores(model, test_ends, X_scaled, window_size)
    test_labels = y_all[test_ends]
    test_attack_ids = attack_id_all[test_ends]

    preds = (test_scores >= threshold_max_val).astype(int)
    _, adjusted_preds = point_adjusted_labels_and_preds(test_labels, preds, test_attack_ids)

    results = {
        "threshold": threshold_max_val,
        "precision": precision_score(test_labels, preds, zero_division=0),
        "recall": recall_score(test_labels, preds, zero_division=0),
        "f1": f1_score(test_labels, preds, zero_division=0),
        "roc_auc": roc_auc_score(test_labels, test_scores) if test_labels.sum() > 0 else float("nan"),
        "pr_auc": average_precision_score(test_labels, test_scores) if test_labels.sum() > 0 else float("nan"),
        "f1_point_adjusted": f1_score(test_labels, adjusted_preds, zero_division=0),
    }

    print(f"[{label}]")
    for k, v in results.items():
        if k != "threshold":
            print(f"  {k:24s}: {v:.4f}")

    print("  Per-attack detection (attacks 5-7):")
    for attack_id in sorted(TEST_ATTACKS):
        mask = test_attack_ids == attack_id
        if mask.sum() == 0:
            continue
        n_labeled = int(test_labels[mask].sum())
        if n_labeled == 0:
            print(f"    Attack {attack_id}: 0 officially labeled hours, detection undefined")
            continue
        detected = bool((preds[mask] & (test_labels[mask] == 1)).any())
        print(f"    Attack {attack_id}: {'DETECTED' if detected else 'MISSED'}")

    return results, test_scores, test_labels, test_attack_ids, preds, threshold_max_val

## 9. GDN and TGCN, Granger graph

In [16]:
print("training GDN (Granger graph)...")
gdn_forecaster = train_forecaster(
    GDNForecaster, {"n_nodes": n_nodes, "edge_index": granger_edge_index, "hidden_dim": 16, "gru_hidden": 64},
    gdn_train_ends, gdn_val_ends, X_scaled_gdn, WINDOW_SIZE
)
gdn_results, gdn_test_scores, gdn_test_labels, gdn_test_attack_ids, gdn_test_preds, gdn_threshold = evaluate_forecaster(
    gdn_forecaster, gdn_val_ends, gdn_test_ends, X_scaled_gdn, y_all, attack_id_all, WINDOW_SIZE,
    label="GDN + Granger-causal graph"
)

training GDN (Granger graph)...
  epoch 1/40 | val forecast MSE 0.49091
  epoch 6/40 | val forecast MSE 0.42385
  epoch 11/40 | val forecast MSE 0.42560
  early stopping at epoch 14
[GDN + Granger-causal graph]
  precision               : 0.7108
  recall                  : 0.7375
  f1                      : 0.7239
  roc_auc                 : 0.9697
  pr_auc                  : 0.7685
  f1_point_adjusted       : 0.9877
  Per-attack detection (attacks 5-7):
    Attack 5: MISSED
    Attack 6: DETECTED
    Attack 7: 0 officially labeled hours, detection undefined


In [17]:
print("training TGCN (Granger graph)...")
tgcn_forecaster = train_forecaster(
    TGCNForecaster, {"n_nodes": n_nodes, "adj_norm": adj_norm_granger, "hidden_dim": 16, "gru_hidden": 64},
    gdn_train_ends, gdn_val_ends, X_scaled_gdn, WINDOW_SIZE
)
tgcn_results, tgcn_test_scores, tgcn_test_labels, tgcn_test_attack_ids, tgcn_test_preds, tgcn_threshold = evaluate_forecaster(
    tgcn_forecaster, gdn_val_ends, gdn_test_ends, X_scaled_gdn, y_all, attack_id_all, WINDOW_SIZE,
    label="TGCN + Granger-causal graph"
)

training TGCN (Granger graph)...
  epoch 1/40 | val forecast MSE 0.55757
  epoch 6/40 | val forecast MSE 0.43524
  epoch 11/40 | val forecast MSE 0.40667
  epoch 16/40 | val forecast MSE 0.41332
  early stopping at epoch 17
[TGCN + Granger-causal graph]
  precision               : 0.7108
  recall                  : 0.7375
  f1                      : 0.7239
  roc_auc                 : 0.9568
  pr_auc                  : 0.7102
  f1_point_adjusted       : 0.9877
  Per-attack detection (attacks 5-7):
    Attack 5: MISSED
    Attack 6: DETECTED
    Attack 7: 0 officially labeled hours, detection undefined


## 10. Attack pattern library

The knowledge graph needs a
model-independent description of each documented attack, using the full duration window from the
official table rather than the sparser `ATT_FLAG` labeling, so that a concealed period still gets
described physically even where BATADAL's own ground truth leaves it unlabeled. Attribution uses each
sensor's raw-value deviation from its normal operating statistics (median and MAD calibrated on
`dataset03`), not any detector's forecast error, so this library stays independent of what any of the
three detectors below actually flag.

In [18]:
def calibrate_robust(values):
    median = np.median(values, axis=0)
    mad = np.median(np.abs(values - median), axis=0) + 1e-6
    return median, mad

def zscore(values, median, mad):
    return (values - median) / mad

def top_k_sensors(z_matrix, k=5):
    mean_z = z_matrix.mean(axis=0)
    top_idx = np.argsort(-mean_z)[:k]
    return [(FEATURE_COLS[j], float(mean_z[j])) for j in top_idx]

normal_median, normal_mad = calibrate_robust(df03[FEATURE_COLS].values)
df04_raw_z = zscore(df04[FEATURE_COLS].values, normal_median, normal_mad)
df04_datetimes = df04["DATETIME"].values

attack_patterns = []
for i, (start, end) in enumerate(ATTACK_DURATION_WINDOWS.values(), start=1):
    mask = (df04_datetimes >= np.datetime64(start)) & (df04_datetimes <= np.datetime64(end))
    if not mask.any():
        print(f"WARNING: no rows found for attack {i}, skipping")
        continue
    top = top_k_sensors(df04_raw_z[mask], k=5)
    peak_local_idx = np.where(mask)[0][np.argmax(df04_raw_z[mask].max(axis=1))]
    attack_patterns.append({
        "attack_id": f"TRAIN_ATTACK_{i}",
        "attack_number": i,
        "start": str(start),
        "end": str(end),
        "description": ATTACK_DESCRIPTIONS.get(i),
        "sibling": SIBLING_ATTACKS.get(i),
        "implicated_sensors": [s for s, _ in top],
        "implicated_scores": [z for _, z in top],
        "peak_datetime": pd.Timestamp(df04_datetimes[peak_local_idx]),
    })

for a in attack_patterns:
    print(a["attack_id"], "top sensors:", a["implicated_sensors"], "| sibling:", a["sibling"])

TRAIN_ATTACK_1 top sensors: ['F_PU4', 'F_PU11', 'P_J280', 'F_PU1', 'P_J317'] | sibling: None
TRAIN_ATTACK_2 top sensors: ['F_PU4', 'P_J280', 'P_J14', 'P_J317', 'P_J269'] | sibling: 1
TRAIN_ATTACK_3 top sensors: ['F_PU4', 'P_J280', 'P_J14', 'P_J269', 'P_J307'] | sibling: None
TRAIN_ATTACK_4 top sensors: ['F_PU4', 'P_J280', 'P_J14', 'P_J307', 'P_J302'] | sibling: 3
TRAIN_ATTACK_5 top sensors: ['F_PU4', 'F_PU6', 'P_J280', 'F_PU1', 'P_J14'] | sibling: None
TRAIN_ATTACK_6 top sensors: ['F_PU6', 'F_PU4', 'P_J280', 'F_PU1', 'P_J14'] | sibling: 5
TRAIN_ATTACK_7 top sensors: ['F_PU4', 'F_PU11', 'P_J280', 'F_PU1', 'P_J256'] | sibling: 6


## 11. Per-model attribution for the three held-out attacks

The main pipeline's `compute_window_scores` pools forecast error across all
sensors into a single scalar per window, which is what detection needs but not what an explanation
needs. `compute_node_errors` below reuses the same trained GDN and TGCN models without that pooling
step, so each flagged window keeps its per-sensor breakdown. Random Forest attribution uses raw-value
z-scores against its own training partition's normal statistics, since it is a snapshot classifier
without a forecast target of its own. Attack 7 has zero officially labeled hours (section 1), so its
detection outcome is reported as undefined for all three detectors rather than as a miss, matching how
`evaluate_forecaster` already treats it.

In [19]:
def compute_node_errors(model, end_indices, X_scaled, window_size, batch_size=256):
    ds = ForecastWindowDataset(X_scaled, end_indices, window_size)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False)
    model.eval()
    errors = []
    with torch.no_grad():
        for window, target in loader:
            pred = model(window.to(device))
            err = torch.abs(pred - target.to(device)).cpu().numpy()
            errors.append(err)
    return np.concatenate(errors, axis=0)

gdn_val_node_errors = compute_node_errors(gdn_forecaster, gdn_val_ends, X_scaled_gdn, WINDOW_SIZE)
gdn_val_normal_mask = y_all[gdn_val_ends] == 0
gdn_median, gdn_mad = calibrate_robust(gdn_val_node_errors[gdn_val_normal_mask])

tgcn_val_node_errors = compute_node_errors(tgcn_forecaster, gdn_val_ends, X_scaled_gdn, WINDOW_SIZE)
tgcn_val_normal_mask = y_all[gdn_val_ends] == 0
tgcn_median, tgcn_mad = calibrate_robust(tgcn_val_node_errors[tgcn_val_normal_mask])

gdn_test_node_errors = compute_node_errors(gdn_forecaster, gdn_test_ends, X_scaled_gdn, WINDOW_SIZE)
gdn_test_node_z = zscore(gdn_test_node_errors, gdn_median, gdn_mad)

tgcn_test_node_errors = compute_node_errors(tgcn_forecaster, gdn_test_ends, X_scaled_gdn, WINDOW_SIZE)
tgcn_test_node_z = zscore(tgcn_test_node_errors, tgcn_median, tgcn_mad)

test_window_datetimes = combined["DATETIME"].values[gdn_test_ends]

MODEL_ATTRIBUTIONS = {"Random Forest": [], "GDN": [], "TGCN": []}

for attack_number in sorted(TEST_ATTACKS):
    n_labeled = int(y_all[episode_test_idx][attack_id_all[episode_test_idx] == attack_number].sum())

    rf_mask = attack_id_all[episode_test_idx] == attack_number
    rf_rows_z = zscore(X_test_raw[rf_mask][:, :len(FEATURE_COLS)], rf_train_median, rf_train_mad)
    rf_top = top_k_sensors(rf_rows_z, k=5)
    rf_peak_local = np.argmax(rf_rows_z.max(axis=1))
    rf_peak_global_idx = np.where(rf_mask)[0][rf_peak_local]
    rf_detected = "undefined" if n_labeled == 0 else bool(test_preds[rf_mask].any())
    MODEL_ATTRIBUTIONS["Random Forest"].append({
        "attack_number": attack_number,
        "attack_id": f"TRAIN_ATTACK_{attack_number}",
        "implicated_sensors": [s for s, _ in rf_top],
        "implicated_scores": [z for _, z in rf_top],
        "peak_datetime": pd.Timestamp(combined.loc[episode_test_idx[rf_mask], "DATETIME"].values[rf_peak_local]),
        "detected": rf_detected,
    })

    win_mask = attack_id_all[gdn_test_ends] == attack_number

    gdn_top = top_k_sensors(gdn_test_node_z[win_mask], k=5)
    gdn_peak_local = np.argmax(gdn_test_node_z[win_mask].max(axis=1))
    gdn_peak_dt = test_window_datetimes[win_mask][gdn_peak_local]
    gdn_detected = "undefined" if n_labeled == 0 else bool(gdn_test_preds[win_mask].any())
    MODEL_ATTRIBUTIONS["GDN"].append({
        "attack_number": attack_number,
        "attack_id": f"TRAIN_ATTACK_{attack_number}",
        "implicated_sensors": [s for s, _ in gdn_top],
        "implicated_scores": [z for _, z in gdn_top],
        "peak_datetime": pd.Timestamp(gdn_peak_dt),
        "detected": gdn_detected,
    })

    tgcn_top = top_k_sensors(tgcn_test_node_z[win_mask], k=5)
    tgcn_peak_local = np.argmax(tgcn_test_node_z[win_mask].max(axis=1))
    tgcn_peak_dt = test_window_datetimes[win_mask][tgcn_peak_local]
    tgcn_detected = "undefined" if n_labeled == 0 else bool(tgcn_test_preds[win_mask].any())
    MODEL_ATTRIBUTIONS["TGCN"].append({
        "attack_number": attack_number,
        "attack_id": f"TRAIN_ATTACK_{attack_number}",
        "implicated_sensors": [s for s, _ in tgcn_top],
        "implicated_scores": [z for _, z in tgcn_top],
        "peak_datetime": pd.Timestamp(tgcn_peak_dt),
        "detected": tgcn_detected,
    })

for model_name, entries in MODEL_ATTRIBUTIONS.items():
    for e in entries:
        print(model_name, e["attack_id"], "detected:", e["detected"], "top sensors:", e["implicated_sensors"])

Random Forest TRAIN_ATTACK_5 detected: True top sensors: ['F_PU4', 'F_PU6', 'P_J280', 'F_PU1', 'P_J14']
Random Forest TRAIN_ATTACK_6 detected: True top sensors: ['F_PU6', 'F_PU4', 'P_J280', 'F_PU1', 'P_J14']
Random Forest TRAIN_ATTACK_7 detected: undefined top sensors: ['F_PU4', 'F_PU11', 'P_J280', 'F_PU1', 'P_J256']
GDN TRAIN_ATTACK_5 detected: True top sensors: ['F_PU6', 'P_J415', 'F_PU7', 'F_PU4', 'F_PU8']
GDN TRAIN_ATTACK_6 detected: True top sensors: ['F_PU6', 'F_PU7', 'F_PU8', 'F_PU4', 'P_J306']
GDN TRAIN_ATTACK_7 detected: undefined top sensors: ['F_PU11', 'F_PU4', 'P_J415', 'F_PU10', 'P_J256']
TGCN TRAIN_ATTACK_5 detected: True top sensors: ['F_PU6', 'F_PU7', 'P_J415', 'F_PU4', 'L_T3']
TGCN TRAIN_ATTACK_6 detected: True top sensors: ['F_PU6', 'F_PU4', 'F_PU7', 'L_T3', 'F_PU5']
TGCN TRAIN_ATTACK_7 detected: undefined top sensors: ['F_PU11', 'F_PU2', 'F_PU4', 'P_J415', 'F_PU10']


## 12. Knowledge graph

Four sources of structure feed the graph, each node and relationship carries a `dataset` property so
this Neo4j instance can hold more than one dataset's graph side by side without collisions:

- Asset nodes, typed by tag prefix (tank, pump, valve, junction).
- PLC nodes and control-logic edges, transcribed directly from the official attack descriptions.
- Correlation and Granger-causal edges between sensors, both graphs pushed so retrieval can draw on
  either.
- Attack-pattern nodes, one per documented attack, linked to their top implicated sensors and, where
  the table states it, to their sibling attack via a `SIMILAR_TO` relationship.

In [20]:
from neo4j import GraphDatabase

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))

def load_graph(tx, dataset):
    if RESET_DATASET:
        tx.run("MATCH (n {dataset: $dataset}) DETACH DELETE n", dataset=dataset)

    for tag in FEATURE_COLS:
        tx.run(
            "MERGE (s:Sensor {tag: $tag, dataset: $dataset}) SET s.asset_type = $asset_type, s.asset_id = $asset_id",
            tag=tag, dataset=dataset, asset_type=ASSET_TYPES[tag], asset_id=ASSET_IDS[tag],
        )

    plc_names = sorted(set(e[0] for e in plc_edges) | set(e[2] for e in plc_edges if e[1] == "SENDS_READING_TO"))
    for plc in plc_names:
        tx.run("MERGE (p:PLC {name: $name, dataset: $dataset})", name=plc, dataset=dataset)

    for src, rel, dst in plc_edges:
        if rel == "SENDS_READING_TO":
            tx.run(
                f"MATCH (a:PLC {{name: $src, dataset: $dataset}}), (b:PLC {{name: $dst, dataset: $dataset}}) "
                f"MERGE (a)-[:{rel}]->(b)",
                src=src, dst=dst, dataset=dataset,
            )
        elif rel == "THRESHOLD_CONTROLS":
            tx.run(
                f"MATCH (a:Sensor {{tag: $src, dataset: $dataset}}), (b:Sensor {{tag: $dst, dataset: $dataset}}) "
                f"MERGE (a)-[:{rel}]->(b)",
                src=src, dst=dst, dataset=dataset,
            )
        else:
            tx.run(
                f"MATCH (a:PLC {{name: $src, dataset: $dataset}}), (b:Sensor {{tag: $dst, dataset: $dataset}}) "
                f"MERGE (a)-[:{rel}]->(b)",
                src=src, dst=dst, dataset=dataset,
            )

    n = len(FEATURE_COLS)
    corr_adj_dense = np.zeros((n, n), dtype=np.float32)
    corr_full = np.nan_to_num(np.corrcoef(X03, rowvar=False), nan=0.0)
    corr_adj_dense[corr_edge_index_np[0], corr_edge_index_np[1]] = np.abs(corr_full[corr_edge_index_np[0], corr_edge_index_np[1]])

    granger_adj_dense = np.zeros((n, n), dtype=np.float32)
    granger_scores = 1.0 - np.nan_to_num(granger_pvals, nan=1.0)
    corr_full = np.nan_to_num(np.corrcoef(X03, rowvar=False), nan=0.0)
    for src, dst in zip(granger_edge_index_np[0], granger_edge_index_np[1]):
        score = granger_scores[src, dst]
        if score <= 0.0:
            score = max(float(np.abs(corr_full[src, dst])), 1e-3)
        granger_adj_dense[src, dst] = score

    for i in range(n):
        for j in range(n):
            if i != j and corr_adj_dense[i, j] > 0.3:
                tx.run(
                    "MATCH (a:Sensor {tag: $t1, dataset: $dataset}), (b:Sensor {tag: $t2, dataset: $dataset}) "
                    "MERGE (a)-[r:CORRELATED_WITH]->(b) SET r.weight = $w",
                    t1=FEATURE_COLS[i], t2=FEATURE_COLS[j], w=float(corr_adj_dense[i, j]), dataset=dataset,
                )
            if i != j and granger_adj_dense[i, j] > 0.0:
                tx.run(
                    "MATCH (a:Sensor {tag: $t1, dataset: $dataset}), (b:Sensor {tag: $t2, dataset: $dataset}) "
                    "MERGE (a)-[r:GRANGER_CAUSES]->(b) SET r.weight = $w",
                    t1=FEATURE_COLS[i], t2=FEATURE_COLS[j], w=float(granger_adj_dense[i, j]), dataset=dataset,
                )

    for a in attack_patterns:
        tx.run(
            "MERGE (p:AttackPattern {attack_id: $aid, dataset: $dataset}) "
            "SET p.attack_number = $num, p.start = $start, p.end = $end, p.description = $description",
            aid=a["attack_id"], dataset=dataset, num=a["attack_number"], start=a["start"], end=a["end"],
            description=a["description"] or "",
        )
        for tag, z in zip(a["implicated_sensors"], a["implicated_scores"]):
            tx.run(
                "MATCH (p:AttackPattern {attack_id: $aid, dataset: $dataset}), (s:Sensor {tag: $tag, dataset: $dataset}) "
                "MERGE (p)-[r:INVOLVES]->(s) SET r.z_score = $z",
                aid=a["attack_id"], tag=tag, dataset=dataset, z=z,
            )
        if a["sibling"]:
            tx.run(
                "MATCH (p:AttackPattern {attack_id: $aid, dataset: $dataset}), "
                "(sib:AttackPattern {attack_id: $sib_id, dataset: $dataset}) "
                "MERGE (p)-[:SIMILAR_TO]->(sib)",
                aid=a["attack_id"], sib_id=f"TRAIN_ATTACK_{a['sibling']}", dataset=dataset,
            )

with driver.session(database=NEO4J_DATABASE) as session:
    session.execute_write(load_graph, DATASET_NAME)
    GRAPH_BOOKMARKS = session.last_bookmarks()

with driver.session(database=NEO4J_DATABASE, bookmarks=GRAPH_BOOKMARKS) as session:
    relationship_counts = session.run(
        "MATCH (s:Sensor)-[r]->(n:Sensor) WHERE s.dataset = $dataset AND n.dataset = $dataset "
        "RETURN type(r) AS relationship, count(r) AS total ORDER BY relationship",
        dataset=DATASET_NAME,
    ).data()

print("graph loaded for dataset:", DATASET_NAME)
for row in relationship_counts:
    print(row["relationship"], row["total"])

graph loaded for dataset: BATADAL
CORRELATED_WITH 146
GRANGER_CAUSES 302
THRESHOLD_CONTROLS 2


## 13. Retrieval

Given a detector's top implicated sensors for a flagged window, this pulls those sensor nodes and their
asset types, correlated and Granger-linked neighbours, any PLC that controls or reads one of them, and
matching `AttackPattern` nodes ranked by sensor overlap. `exclude_attack_id` lets a query skip
retrieving its own pattern, so cross-attack matching through the documented `SIMILAR_TO` siblings is
what actually gets exercised.

In [21]:
def retrieve_subgraph(implicated_tags, dataset, top_k_patterns=2, exclude_attack_id=None):
    with driver.session(database=NEO4J_DATABASE, bookmarks=GRAPH_BOOKMARKS) as session:
        sensors = session.run(
            "MATCH (s:Sensor) WHERE s.tag IN $tags AND s.dataset = $dataset "
            "RETURN s.tag AS tag, s.asset_type AS asset_type, s.asset_id AS asset_id",
            tags=implicated_tags, dataset=dataset,
        ).data()

        neighbors = session.run(
            "MATCH (s:Sensor)-[r:CORRELATED_WITH|GRANGER_CAUSES]->(n:Sensor) "
            "WHERE s.tag IN $tags AND s.dataset = $dataset AND n.dataset = $dataset "
            "RETURN s.tag AS from_tag, n.tag AS to_tag, n.asset_type AS to_type, type(r) AS relationship, r.weight AS weight "
            "ORDER BY r.weight DESC LIMIT 15",
            tags=implicated_tags, dataset=dataset,
        ).data()

        plcs = session.run(
            "MATCH (p:PLC)-[rel]->(s:Sensor) WHERE s.tag IN $tags AND s.dataset = $dataset AND p.dataset = $dataset "
            "RETURN p.name AS plc, type(rel) AS relationship, s.tag AS sensor",
            tags=implicated_tags, dataset=dataset,
        ).data()

        patterns = session.run(
            "MATCH (p:AttackPattern)-[r:INVOLVES]->(s:Sensor) "
            "WHERE s.tag IN $tags AND s.dataset = $dataset AND p.dataset = $dataset AND p.attack_id <> $exclude "
            "WITH p, count(DISTINCT s) AS overlap, collect(DISTINCT s.tag) AS overlapping_sensors "
            "RETURN p.attack_id AS attack_id, p.description AS description, overlap, overlapping_sensors "
            "ORDER BY overlap DESC LIMIT $k",
            tags=implicated_tags, dataset=dataset, exclude=exclude_attack_id or "", k=top_k_patterns,
        ).data()

    return {"sensors": sensors, "neighbors": neighbors, "plcs": plcs, "matched_patterns": patterns}

retrieve_subgraph(
    MODEL_ATTRIBUTIONS["GDN"][0]["implicated_sensors"],
    dataset=DATASET_NAME,
    exclude_attack_id=MODEL_ATTRIBUTIONS["GDN"][0]["attack_id"],
)

{'sensors': [{'tag': 'F_PU4', 'asset_type': 'pump', 'asset_id': 'PU4'},
  {'tag': 'F_PU6', 'asset_type': 'pump', 'asset_id': 'PU6'},
  {'tag': 'F_PU7', 'asset_type': 'pump', 'asset_id': 'PU7'},
  {'tag': 'F_PU8', 'asset_type': 'pump', 'asset_id': 'PU8'},
  {'tag': 'P_J415', 'asset_type': 'junction', 'asset_id': 'J415'}],
 'neighbors': [{'from_tag': 'F_PU4',
   'to_tag': 'L_T2',
   'to_type': 'tank',
   'relationship': 'GRANGER_CAUSES',
   'weight': 1.0},
  {'from_tag': 'F_PU4',
   'to_tag': 'F_PU8',
   'to_type': 'pump',
   'relationship': 'GRANGER_CAUSES',
   'weight': 1.0},
  {'from_tag': 'F_PU7',
   'to_tag': 'L_T3',
   'to_type': 'tank',
   'relationship': 'GRANGER_CAUSES',
   'weight': 1.0},
  {'from_tag': 'F_PU4',
   'to_tag': 'L_T4',
   'to_type': 'tank',
   'relationship': 'GRANGER_CAUSES',
   'weight': 1.0},
  {'from_tag': 'P_J415',
   'to_tag': 'F_PU7',
   'to_type': 'pump',
   'relationship': 'GRANGER_CAUSES',
   'weight': 1.0},
  {'from_tag': 'F_PU4',
   'to_tag': 'L_T3',
 

## 14. Explanation generation with Groq

In [22]:
from groq import Groq

groq_client = Groq(api_key=GROQ_API_KEY)
GROQ_MODEL = "openai/gpt-oss-120b"

def build_prompt(model_name, entry, subgraph):
    sensor_lines = "\n".join(
        f"- {tag} (z-score {z:.1f})" for tag, z in zip(entry["implicated_sensors"], entry["implicated_scores"])
    )
    type_lookup = {s["tag"]: s["asset_type"] for s in subgraph["sensors"]}
    type_lines = "\n".join(f"- {tag}: {type_lookup.get(tag, 'unknown')}" for tag in entry["implicated_sensors"])
    neighbor_lines = "\n".join(
        f"- {n['from_tag']} <-> {n['to_tag']} ({n['to_type']}), {n['relationship']} weight {n['weight']:.2f}"
        for n in subgraph["neighbors"][:8]
    ) or "(none retrieved)"
    plc_lines = "\n".join(
        f"- {p['plc']} {p['relationship']} {p['sensor']}" for p in subgraph["plcs"]
    ) or "(no PLC relationships retrieved for these sensors)"
    if subgraph["matched_patterns"]:
        pattern_lines = "\n".join(
            f"- {p['attack_id']} shares sensors {p['overlapping_sensors']}"
            + (f", documented description: {p['description']}" if p["description"] else "")
            for p in subgraph["matched_patterns"]
        )
    else:
        pattern_lines = "(no matching historical attack pattern found)"

    detection_line = (
        "not evaluated, this attack has zero officially labeled hours in the ground truth"
        if entry["detected"] == "undefined"
        else ("flagged as anomalous" if entry["detected"] else "not flagged as anomalous under the calibrated threshold")
    )

    return f"""You are a security analyst assistant for a water distribution SCADA system (BATADAL / C-Town network).
The {model_name} detector's evidence is centred around {entry['peak_datetime']}, this window was {detection_line}.
Explain it for a control-room analyst.

Most implicated sensors, ranked by the detector's own attribution:
{sensor_lines}

Asset types of those sensors:
{type_lines}

PLC control-logic relationships involving these sensors:
{plc_lines}

Sensors correlated or Granger-linked with the implicated ones under normal operation:
{neighbor_lines}

Matching historical attack pattern(s) from documented training data:
{pattern_lines}

Write a concise (4 to 6 sentence) explanation covering: (1) which assets are implicated and what kind of assets
they are, (2) any relevant PLC control chain, (3) whether this resembles a documented historical attack pattern
and why, (4) a plausible hypothesis for what is physically happening. Only use the information given above, do
not invent sensor names, PLC relationships, or attack details that were not provided."""

def generate_explanation(model_name, entry, subgraph):
    prompt = build_prompt(model_name, entry, subgraph)
    response = groq_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
        max_tokens=600,
    )
    return response.choices[0].message.content

explanations = []
for model_name, entries in MODEL_ATTRIBUTIONS.items():
    for entry in entries:
        subgraph = retrieve_subgraph(
            entry["implicated_sensors"], dataset=DATASET_NAME, exclude_attack_id=entry["attack_id"]
        )
        text = generate_explanation(model_name, entry, subgraph)
        explanations.append({"model": model_name, "entry": entry, "subgraph": subgraph, "explanation": text})
        print(f"--- {model_name}, attack {entry['attack_number']} ({entry['peak_datetime']}) ---")
        print(text)
        print()

--- Random Forest, attack 5 (2016-11-26 23:00:00) ---
The Random Forest detector flags an anomaly centered on 2016‑11‑26 23:00 UTC, driven almost entirely by three pumps (F_PU4, F_PU6, F_PU1) and two junctions (P_J280, P_J14). PLC 1 is the only explicit control link in the evidence, directly commanding pump F_PU1, which under normal conditions is tightly coupled (Granger‑causal) with the other pumps, junctions and tanks (L_T1, L_T2). The sensor set matches the documented pattern TRAIN_ATTACK_6 (which involved the same five assets) and also overlaps with TRAIN_ATTACK_2, suggesting the attacker is using a known “pump‑speed reduction + replay concealment” technique. A plausible physical scenario is that the attacker abruptly reduced the speed or flow of pumps F_PU4, F_PU6 (and possibly F_PU1) to force a rapid drop in downstream tank levels (e.g., L_T4), while replaying historic sensor data to hide the abnormal flow and level readings at the junctions.

--- Random Forest, attack 6 (2016-12

## 15. Explanation quality check

A lightweight, programmatic version of the rubric the proposal describes: does each explanation
name-check the true implicated sensors and assets it was given, does it correctly flag a documented
sibling match only when one was genuinely retrieved rather than hallucinated, and did retrieval actually
surface the documented sibling attack where one exists. This is not a substitute for a human qualitative
read in the dissertation write-up, but it catches the cheap failure modes automatically, and it lets the
three detectors be compared on identical grounds.

In [23]:
def check_explanation(item):
    text = item["explanation"].lower()
    entry = item["entry"]
    true_sensors = entry["implicated_sensors"]
    true_assets = sorted(set(ASSET_IDS[t] for t in true_sensors))

    sensor_mentions = sum(1 for t in true_sensors if t.lower() in text)
    asset_mentions = sum(1 for a in true_assets if a.lower() in text)

    matched_ids = {p["attack_id"] for p in item["subgraph"]["matched_patterns"]}
    sibling_number = SIBLING_ATTACKS.get(entry["attack_number"])
    expected_sibling_id = f"TRAIN_ATTACK_{sibling_number}" if sibling_number else None
    sibling_retrieved = expected_sibling_id in matched_ids if expected_sibling_id else None

    claims_pattern = any(kw in text for kw in ["historical", "resembl", "similar to", "matches", "documented"])

    return {
        "model": item["model"],
        "attack_number": entry["attack_number"],
        "detected": entry["detected"],
        "sensors_mentioned": f"{sensor_mentions}/{len(true_sensors)}",
        "assets_mentioned": f"{asset_mentions}/{len(true_assets)}",
        "documented_sibling": sibling_number,
        "sibling_correctly_retrieved": sibling_retrieved,
        "pattern_claimed_in_text": claims_pattern,
        "consistent": (len(matched_ids) > 0) or (not claims_pattern),
    }

quality_df = pd.DataFrame([check_explanation(e) for e in explanations])
quality_df

,model,attack_number,detected,sensors_mentioned,assets_mentioned,documented_sibling,sibling_correctly_retrieved,pattern_claimed_in_text,consistent
0,Random Forest,5,True,5/5,5/5,NaN,None,True,True
1,Random Forest,6,True,5/5,5/5,5.0,True,True,True
2,Random Forest,7,undefined,5/5,5/5,6.0,False,True,True
3,GDN,5,True,5/5,5/5,NaN,None,True,True
4,GDN,6,True,5/5,5/5,5.0,True,True,True
5,GDN,7,undefined,5/5,5/5,6.0,False,True,True
6,TGCN,5,True,5/5,5/5,NaN,None,True,True
7,TGCN,6,True,5/5,5/5,5.0,True,True,True
8,TGCN,7,undefined,5/5,5/5,6.0,False,True,True


## Notes

grangercausalitytests drops its verbose argument in statsmodels 0.15. An older environment
passing it would raise on every pairwise test and silently fall back every node to correlation
edges, since the call sits inside a broad try/except. Section 5 prints the fallback count;
check it's a small minority before treating the graph as genuinely causal.

PLC-level relationships are transcribed from the attack table's description column (PLC1, PLC2,
PLC9 only), not a full C-Town network diagram.

Retrieval is exact structural overlap on a small graph, not embedding-based.

Attack 7 has zero officially labeled hours, so its detection outcome is reported as undefined
for all three detectors, not a miss. An explanation is still generated from its highest-evidence
window since the attack physically occurred regardless of official labeling.

Neo4j Aura's free tier pauses on inactivity; re-run the driver connection cell after a break.